In this optional session, I will briefly introduce **Object-Oriented Programming (OOP)** in Python.

**Learning Outcomes**

- What Object-Oriented Programming is and how it is used in AccFin research
- How to create and use **classes** and **objects**
- Core OOP concepts: **attributes** and **methods**, **encapsulation**
- How to understand various python instances from the perspecitve of OOP

### 1. Introduction to OOP and Basic Terminology

In Python, **everything** you create — a number, a string, a DataFrame — is an **object**.
A **class** is the blueprint that defines what an object looks like and how it behaves: what data it stores (its **attributes**) and what it can do (its **methods**).

Suppose we want to represent one firm-year observation from a Compustat-style panel — one row identified by a `gvkey`/`fyear` pair, together with its financial variables. Instead of juggling `gvkey`, `fyear`, `at`, etc. as separate loose variables for every firm, we can define a `FirmYear` **class** once as a blueprint, then create one **instance** — one concrete `FirmYear` **object** — per row of data.

- **Attributes**: data stored on the object (e.g., `gvkey`, `year`, `at`).
- **Methods**: functions defined inside the class that describe the object's behavior (e.g., computing a ratio, printing a summary).
- **Instantiation**: creating an object from a class. When you do this, the object becomes an **instance** of that class.

The rest of this notebook builds up a `FirmYear` class piece by piece to illustrate these ideas.

### 2. Creating Classes and Objects

#### 2.1 The `__init__` Method and the `self` Parameter

The `__init__` method is a special method that Python runs automatically whenever you create ("instantiate") a new object from a class. It's where you set up the object's initial attributes.

Every instance method — including `__init__` — takes `self` as its first parameter. `self` refers to *this particular* instance, so `self.gvkey = gvkey` means "store `gvkey` on this specific `FirmYear` object."

In [ ]:
class FirmYear:
    def __init__(self, gvkey: str, fyear: int):
        self.gvkey = gvkey # Attributes
        self.fyear = fyear

    def show_info(self): # Method
        print(f"This is the observation for gvkey {self.gvkey} in fiscal year {self.fyear}.")

#### 2.2 Creating an Object (Instance)

Calling the class like a function — `FirmYear(1004, 2024)` — runs `__init__` behind the scenes and returns a new, independent object.

In [ ]:
TSLA24 = FirmYear("184996", 2024)

In [ ]:
# Attribute
print(TSLA24.gvkey)
print(TSLA24.fyear)

# Method
TSLA24.show_info()

In [ ]:
# Each instance keeps its own attributes, independent of other instances
TSLA23 = FirmYear(1010, 2023)
TSLA23.show_info()

print(TSLA23.gvkey == TSLA24.gvkey)

#### 2.3 Object Relationships: Composing an `Industry` into `FirmYear`

Classes can be related to one another in their attributes.

A firm belongs to an industry, so it's natural to model `Industry` as its own class and pass an `Industry` object in as an attribute of `FirmYear`. This is called **composition** — a "has-a" relationship (a `FirmYear` *has an* `Industry`), as opposed to the "is-a" relationship you'll see later with inheritance.

In [ ]:
class Industry:
    def __init__(self, sic_code: int, name: str):
        assert isinstance(sic_code, int), "SIC code must be an integer"
        assert len(str(sic_code)) == 4, "SIC code must be a 4-digit integer" # check the format of inputs before creating the object
        self.sic_code = sic_code
        self.name = name

    def info(self):
        print(f"Industry {self.sic_code}: {self.name}")

In [ ]:
class FirmYear:
    def __init__(self, gvkey: str, fyear: int, industry: Industry):
        self.gvkey = gvkey
        self.fyear = fyear
        self.industry = industry   # an Industry object, not just a string or code

    def info(self):
        print(f"gvkey {self.gvkey}, fiscal fyear {self.fyear}, industry: {self.industry.name}")

In [ ]:
tech = Industry(7372, "Prepackaged Software")
MSFT26 = FirmYear("012141", 2026, tech)

MSFT26.info()
print(MSFT26.industry.sic_code)   # reach through FirmYear to its Industry object

### 3. Access Modifiers (Public, Protected, and Private)

Python doesn't strictly enforce access control the way some other languages (Java, C++) do — it relies on a **"consenting adults"** philosophy: conventions signal intent, and it's up to the developer to respect them.

- A single leading underscore (`_total_assets`) marks an attribute as **protected**: a signal that it's intended for internal use within the class and shouldn't be accessed directly from the outside — even though nothing technically stops you from doing so.
- A double leading underscore (`__total_assets`) marks an attribute as **private**. Python performs **name mangling** on these behind the scenes — `__total_assets` is actually stored as `_FirmYear__total_assets` — which makes it awkward (though still not impossible) to access from outside the class.

In [ ]:
class FirmYear:
    def __init__(self, gvkey, fyear, at):
        self.gvkey = gvkey
        self.year = fyear
        self._total_assets = at   # protected: a convention, not enforced

f1 = FirmYear(1004, 2024, 5000)
print(f1._total_assets)   # still accessible, but the underscore signals "please don't"

In [ ]:
class FirmYear:
    def __init__(self, gvkey, year, total_assets):
        self.gvkey = gvkey
        self.year = year
        self.__total_assets = total_assets   # private: name-mangled

f1 = FirmYear(1004, 2024, 5000)

try:
    print(f1.__total_assets)
except AttributeError as e:
    print(f"Error: {e}")

print(f1._FirmYear__total_assets)   # name mangling: the "real" attribute name underneath

### 4. Getters, Setters, and Properties

If an attribute is protected or private, how do we let users read or update it safely? The traditional approach is to write explicit `get_...`/`set_...` methods that add validation logic — e.g., a firm's total assets should never be negative.

In [ ]:
class FirmYear:
    def __init__(self, gvkey, year, at):
        self.gvkey = gvkey
        self.year = year
        self.__total_assets = at

    def get_total_assets(self):
        return self.__total_assets

    def set_total_assets(self, at):
        if at < 0:
            raise ValueError("Total Assets cannot be negative.")
        self.__total_assets = at

f1 = FirmYear(1004, 2024, 5000)
print(f1.get_total_assets())

f1.set_total_assets(5200)
print(f1.get_total_assets())

f1.set_total_assets(-100)   # This will raises ValueError

This works, but calling `f1.get_total_assets()` / `f1.set_total_assets(...)` is clunkier than plain dot notation. Python's `@property` decorator lets you keep the same validation logic while still writing `f1.total_assets`, exactly as if it were a public attribute. This is the **recommended** approach in Python.

We can also add a `roa` (return on assets) **property** that's computed on the fly from the object's current attributes, rather than stored — so it's always consistent with the latest data.

In [ ]:
class FirmYear:
    def __init__(self, gvkey, fyear, at, ni):
        self.gvkey = gvkey
        self.fyear = fyear
        self.__total_assets = at
        self.__net_income = ni

    @property
    def total_assets(self):
        return self.__total_assets

    @total_assets.setter
    def total_assets(self, at):
        if at < 0:
            raise ValueError("Total Assets cannot be negative.")
        self.__total_assets = at

    @property
    def roa(self):
        """Return on assets, computed on the fly from current attributes."""
        return self.__net_income / self.__total_assets

f1 = FirmYear("1004", 2024, at=5000, ni=450)
print(f1.total_assets)     # calls the getter, looks like a normal attribute
print(f1.roa)

f1.total_assets = 5200     # calls the setter, validation runs automatically
print(f1.roa)              # roa updates automatically because it's computed, not stored

f1.total_assets = -1       # raises ValueError

### 5. Static vs. Instance Attributes and Methods

- **Instance attributes/methods** (using `self`) are unique to each individual object — perfect for storing object-specific data, like one firm-year's `gvkey` or `total_assets`. Instance methods have access to these specific attributes.
- **Static (class) attributes** are stored at the class level and shared among all instances. There's only one copy in memory, which makes them ideal for things like a running counter of how many objects have been created.
- **Static methods** (`@staticmethod`) belong to the class rather than any instance. They don't take `self` and are useful for utility functions that don't depend on any particular object's data — like checking whether a discount rate is a valid input.

In [ ]:
class FirmYear:
    num_observations = 0   # static/class attribute: shared by every instance

    def __init__(self, gvkey, fyear, at):
        self.gvkey = gvkey
        self.fyear = fyear
        self.total_assets = at
        FirmYear.num_observations += 1   # update the shared, class-level counter

    @staticmethod
    def is_valid_discount_rate(rate):
        """Utility check that doesn't depend on any particular FirmYear instance."""
        return 0 <= rate <= 1

f1 = FirmYear(1004, 2024, 5000)
f2 = FirmYear(1010, 2023, 3200)

print(FirmYear.num_observations)          # 2, shared across all instances
print(FirmYear.is_valid_discount_rate(0.08))
print(FirmYear.is_valid_discount_rate(1.5))

### 6. Core OOP Principles

Everything so far has been building toward four core ideas: **encapsulation**, **abstraction**, **inheritance**, and **polymorphism**.

#### 6.1 Encapsulation

Encapsulation is the practice of bundling data and methods into a single class, and restricting direct access to the internal implementation details. A classic example: a general ledger account's cash balance shouldn't be edited directly — every change should go through a controlled `deposit`/`withdraw` method that enforces valid accounting logic (e.g., you can't overdraw the account).

In [ ]:
class GeneralLedgerAccount:
    def __init__(self, opening_balance: float = 0.0):
        self.__balance = opening_balance   # hidden: no direct access from outside

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.__balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdraw amount must be positive.")
        if amount > self.__balance:
            raise ValueError("Insufficient funds: cannot overdraw the account.")
        self.__balance -= amount

cash = GeneralLedgerAccount(opening_balance=10000)
cash.deposit(2500)     # e.g., cash receipt from a customer
cash.withdraw(4000)    # e.g., payment to a supplier
print(cash.balance)

cash.withdraw(100000)  # This will raises ValueError: balance is protected from going negative

#### 6.2 Abstraction

Abstraction means hiding complexity behind a simplified, high-level interface. Think of connecting to WRDS: behind the scenes there's connecting, authenticating, running a query, and disconnecting — but the user should just be able to ask for the data they want.

In [ ]:
class WRDSDataService:
    def __init__(self, username):
        self.username = username

    def __connect(self):
        print(f"Connecting to WRDS as {self.username}...")

    def __authenticate(self):
        print("Authenticating credentials...")

    def __run_query(self, gvkey, fyear):
        print(f"Querying Compustat for gvkey {gvkey}, fiscal year {fyear}...")
        return {"gvkey": gvkey, "year": fyear, "total_assets": 5000}

    def __disconnect(self):
        print("Closing WRDS connection.")

    def get_financials(self, gvkey, year):
        """The only method a user actually needs to call."""
        self.__connect()
        self.__authenticate()
        data = self.__run_query(gvkey, year)
        self.__disconnect()
        return data

service = WRDSDataService(username="leonard")
result = service.get_financials(gvkey="1004", year=2024)
print(result)

The user only ever calls `get_financials(...)` — the connect/authenticate/query/disconnect steps are hidden. In Module 3, you'll use the real `wrds` package the same way: call `db.raw_sql(...)` without worrying about the connection details underneath.

#### 6.3 Inheritance

Inheritance lets a new subclass inherit properties and behaviors from an existing parent (super)class, modeling an "is a" relationship — a `Stock` *is a* `Security`, and so is a `Bond`. Subclasses use `super()` to reuse the parent's setup instead of duplicating it.

In [ ]:
class Security:
    def __init__(self, ticker: str, issuer: str):
        self.ticker = ticker
        self.issuer = issuer

    def describe(self):
        print(f"{self.ticker} issued by {self.issuer}")


class Stock(Security):
    def __init__(self, ticker, issuer, dividend_yield):
        super().__init__(ticker, issuer)   # reuse Security's __init__ instead of repeating it
        self.dividend_yield = dividend_yield


class Bond(Security):
    def __init__(self, ticker, issuer, coupon_rate, face_value):
        super().__init__(ticker, issuer)
        self.coupon_rate = coupon_rate
        self.face_value = face_value

In [ ]:
aapl = Stock(ticker="AAPL", issuer="Apple Inc.", dividend_yield=0.005)
t10y = Bond(ticker="T10Y", issuer="US Treasury", coupon_rate=0.04, face_value=1000)

aapl.describe()  # inherited from Security, not redefined in Stock
t10y.describe()  # inherited from Security, not redefined in Bond

print(isinstance(aapl, Security), isinstance(t10y, Security))   # a Stock/Bond "is a" Security

#### 6.4 Polymorphism

Polymorphism means "having many forms": different object types can be treated uniformly through a shared parent class. A program can loop through a mix of `Stock` and `Bond` objects and call the same method — e.g., `expected_return()` — dynamically running each subclass's own overridden implementation, with no messy `if`/`elif` chain needed.

In [ ]:
class Stock(Security):
    def __init__(self, ticker, issuer, dividend_yield):
        super().__init__(ticker, issuer)
        self.dividend_yield = dividend_yield

    def expected_return(self):
        return self.dividend_yield + 0.07   # dividend yield + an assumed capital gain

class Bond(Security):
    def __init__(self, ticker, issuer, coupon_rate, face_value):
        super().__init__(ticker, issuer)
        self.coupon_rate = coupon_rate
        self.face_value = face_value

    def expected_return(self):
        return self.coupon_rate   # simplified: yield to maturity ~= coupon rate at par

portfolio = [
    Stock(ticker="AAPL", issuer="Apple Inc.", dividend_yield=0.005),
    Bond(ticker="T10Y", issuer="US Treasury", coupon_rate=0.04, face_value=1000),
    Stock(ticker="MSFT", issuer="Microsoft Corp.", dividend_yield=0.008),
]

for security in portfolio:
    # same method call, different behavior depending on the actual object type
    print(f"{security.ticker}: {security.expected_return():.2%}")

Notice:
- there's no 
```python
if isinstance(security, Stock): 
    ... 
elif isinstance(security, Bond):
    ...
```
chain needed in the loop above. Each object simply knows how to compute its own `expected_return()`.
- Polymorphism does not require two classes to inherit from the same parent class.
